# 第 2 周第 4 天作业 —— 航班票价工具调用

### 练习目标

做一个航班预订助手 **FlightAI**：既能回答问题，又能通过 **function calling / tools** 读写票价。

票价存在本地 **SQLite** 数据库 `prices.db` 里；模型可调用：

- `get_ticket_price`：查某城市往返票价
- `set_ticket_price`：为某城市设置/更新票价

### 和本课概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Tool / Function Calling | `tools=` + `finish_reason=="tool_calls"` |
| 工具 schema | JSON 描述 name / parameters |
| 工具执行循环 | 调工具 → 把 tool 结果塞回 messages → 再问模型 |
| 本地 Ollama | OpenAI 兼容客户端 + `llama3.2` |

### 怎么跑

1. 启动 Ollama 并确保有 `llama3.2`
2. 依次运行：建库与函数 → 工具定义 → 处理函数 → chat → launch
3. 在聊天里试：「东京票价多少？」「把巴黎票价改成 999」


In [ ]:
# ========== 导入 ==========

# json：把模型返回的 tool arguments（JSON 字符串）解析成 Python dict
import json
# OpenAI SDK：对接本地 Ollama 的兼容接口
from openai import OpenAI
# Gradio：聊天界面
import gradio as gr


In [ ]:
# ========== 模型与客户端 ==========

# 本地模型名（须已 ollama pull）
MODEL = "llama3.2"
# 变量名 openai：客户端实例，实际指向本机 Ollama（不是云端 OpenAI）
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


In [ ]:
# ========== system prompt：航空助手人设 ==========
# 英文原文保留：这是发给模型的系统指令

system_message = """
You are a helpful assistant for an Airline called FlightAI.
You can get and update real-time price of tickets.
Give short, courteous answers.
Always be accurate. If you don't know the answer, say so.
"""


In [ ]:
# ========== SQLite 票价库 + 两个「真实」工具函数 ==========
# 用初始价格填充数据库，并定义要作为 tool 调用的 Python 函数

# sqlite3：内置轻量数据库，适合本练习把票价落盘
import sqlite3
# 数据库文件名（相对当前工作目录）
DB = "prices.db"
# 连接（with 结束会自动关闭）；没有表就建 prices(city, price)
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    # city 主键：每个城市一行；price 用 REAL（浮点）
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

def get_ticket_price(city):
    # 调试日志：确认模型真的触发了工具（flush 立刻打到终端）
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        # 参数化查询：city 统一 lower，避免大小写不一致
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        # 查到就格式化返回；查不到给明确提示（这些返回字符串会作为 tool 消息回给模型）
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

def set_ticket_price(city, price):
    # 调试日志：写入路径被调用时打印
    print(f"DATABASE TOOL CALLED: Setting price for {city}: ${price}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        # UPSERT：没有则插入，有冲突（同 city）则更新 price
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()
    return f"Ticket price to {city} is ${price}"

# 种子数据：几个示例城市的初始票价
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
# 逐个写入数据库，方便一开聊就能查到数
for city, price in ticket_prices.items():
    set_ticket_price(city, price)


In [ ]:
# ========== 工具 schema：告诉模型「有哪些函数、参数长什么样」 ==========
# 注意：这里是 JSON 描述，不是 Python 函数本身；name 必须和真正可调用的函数对应

get_price_function = {
    # 工具名：模型会在 tool_calls 里带回这个名字
    "name": "get_ticket_price",
    # description：模型靠这段文字判断何时该调用
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        # 必填参数列表
        "required": ["destination_city"],
        # 不允许额外未知字段
        "additionalProperties": False
    }
}

set_price_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
            "price": {
                "type": "number",
                "description": "The ticket price for the city",
            },
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False
    }
}


In [ ]:
# ========== 组装 tools 列表：传给 chat.completions.create(tools=...) ==========

# OpenAI / 兼容 API 要求：每项带 type="function"，内层再放 function schema
tools = [
    {"type": "function", "function": get_price_function},
    {"type": "function", "function": set_price_function},
]


In [ ]:
# ========== 执行工具调用：把模型的 tool_calls 变成 role=tool 的消息 ==========

def handle_tool_calls(message):
    # 收集本次所有工具结果，稍后 extend 进 messages
    responses = []
    # 名字 → 真正的 Python 函数（与 schema 的 name 对齐）
    operations = {
        'get_ticket_price': get_ticket_price,
        'set_ticket_price': set_ticket_price,
    }
    # 一次回复里可能有多个 tool_call，逐个处理
    for tool_call in message.tool_calls:
        if tool_call.function.name in operations:
            # arguments 是 JSON 字符串，先 loads 成 dict
            arguments = json.loads(tool_call.function.arguments)
            # schema 里参数叫 destination_city；Python 函数形参叫 city
            city = arguments.get('destination_city')
            price = arguments.get('price')
            # 有 price 就走 set；没有就走 get（依赖「set 必带 price」的约定）
            if price:
                price_details = operations[tool_call.function.name](city, price)
            else:
                price_details = operations[tool_call.function.name](city)
            # 工具结果必须带回 tool_call_id，模型才能对上是哪次调用
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        else:
            # 未知工具名：打印并仍返回一条 tool 消息，避免对话链断裂
            print(f"Unknown operation: {tool_call.function.name}")
            responses.append({
                "role": "tool",
                "content": f"Unknown operation: {tool_call.function.name}",
                "tool_call_id": tool_call.id
            })
    return responses


In [ ]:
# ========== 对话主循环：支持多轮 tool calling ==========

def chat(message, history):
    # 把 Gradio history 规范成 OpenAI messages 片段
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # system + 历史 + 当前用户
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 第一次请求：带上 tools，让模型决定是否调用
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    # 若 finish_reason 是 tool_calls：执行工具 → 追加结果 → 再问模型，直到不再要工具
    while response.choices[0].finish_reason=="tool_calls":
        # 助手那条带 tool_calls 的消息
        message = response.choices[0].message
        # 本地执行 get/set
        responses = handle_tool_calls(message)
        # 先把助手消息（含 tool_calls）放进上下文
        messages.append(message)
        # 再追加所有 role=tool 的结果
        messages.extend(responses)
        # 带着工具结果再调用一次（仍声明 tools，以便连环调用）
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    # 最终自然语言回复
    return response.choices[0].message.content


In [ ]:
# ========== 启动 Gradio 聊天界面 ==========

# type="messages"：history 使用 [{role, content}, ...] 格式，和上面 chat 一致
gr.ChatInterface(fn=chat, type="messages").launch()
